# 03 · Train (or fit) the dynamics model

We fit the classical encoder, then evaluate one-step-ahead trajectory RMSE under the prior dynamics.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt


In [ ]:
from data.synthetic_generator import generate_synthetic_cohort
from features import engineer_all_features
from states.latent_state_encoder import encode_latent_states_classical, LATENT_DIM_NAMES
from dynamics.transition_model import LatentDynamicsModel
from evaluation.metrics import trajectory_rmse

df = engineer_all_features(generate_synthetic_cohort(n_participants=60, n_days=120, seed=17))
df = df.sort_values(['participant_id','date']).reset_index(drop=True)


In [ ]:
def pick(df, cols):
    present = [c for c in cols if c in df.columns]
    return df[present].to_numpy(dtype=float, na_value=0.0) if present else np.zeros((len(df), len(cols)))

W = pick(df, ['sleep_duration_hours','hrv_rmssd','resting_hr','daily_steps','recovery_score'])
B = pick(df, ['screen_time_minutes','mobility_radius_km','location_entropy','phone_unlock_count'])
C = pick(df, ['temperature_c','nighttime_temperature_c','aqi','heat_wave_flag'])
M = pick(df, ['missing_wearable_flag','missing_phone_flag','missing_survey_flag'])
P = pick(df, ['baseline_hrv','baseline_resting_hr','baseline_climate_vulnerability','baseline_resilience'])
Z = encode_latent_states_classical(W, B, C, M, P).latent


In [ ]:
dynamics = LatentDynamicsModel(rng_seed=17)
same_pid = df['participant_id'].values[1:] == df['participant_id'].values[:-1]
Z_t, Z_next = Z[:-1][same_pid], Z[1:][same_pid]
env_cols = ['temperature_c','nighttime_temperature_c','aqi','heat_wave_flag']
beh_cols = ['screen_time_minutes','mobility_radius_km','location_entropy','phone_unlock_count']
E_t = df[env_cols].to_numpy(dtype=float, na_value=0.0)[:-1][same_pid]
B_t = df[beh_cols].to_numpy(dtype=float, na_value=0.0)[:-1][same_pid]
preds = np.stack([dynamics.predict_next_state(z, b, e) for z, b, e in zip(Z_t, B_t, E_t)])
rmse = trajectory_rmse(Z_next, preds)
print(f'One-step latent RMSE: {rmse:.4f}')
